# Đóng gói thư mục `data` trên Google Drive thành `data.tar`

Notebook này chạy trên Colab: nén thư mục `data` của nhóm trên Drive thành một tệp `data.tar` và đặt cạnh nó trong `DRIVE_PROJECT_DIR`, đúng chỗ `parser.ipynb` tự tìm (`data.tar.gz` → `data.tgz` → `data.tar`).

Các bước:
1. Mount Drive, kiểm tra thư mục nguồn, đếm số tệp và dung lượng.
2. Tạo tệp tar trên **ổ local** của runtime (`/content`), vì ghi trực tiếp một tệp lớn lên Drive qua FUSE chậm và dễ hỏng giữa chừng.
3. Kiểm tra tệp tar (đếm số mục, so với số tệp nguồn).
4. Chép lên Drive dưới tên tạm `*.partial` rồi đổi tên, nên `parser.ipynb` không bao giờ thấy một tệp chép dở.

Tệp tar chứa thư mục `data/` ở gốc (giống bản `data.tar` hiện có), nên giải nén ra `.../data/<paper_id>/...`.

**Lưu ý:** tệp tar là ảnh chụp tại thời điểm tạo; khi dữ liệu trên Drive thay đổi thì chạy lại notebook này.

## 1. Cấu hình

In [ ]:
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intro2DS-Group3"  # thư mục chứa `data/`
SOURCE_NAME = "data"            # tên thư mục cần nén, nằm trong DRIVE_PROJECT_DIR
COMPRESS = False                # False -> data.tar; True -> data.tar.gz (nhỏ hơn nhưng nén/giải nén chậm hơn)
EXCLUDE = []                    # mẫu bỏ qua, ví dụ [".ipynb_checkpoints", "*.tmp"]
LOCAL_WORK_DIR = "/content"     # ổ local của runtime, nơi tạo tar trước khi chép lên Drive
OVERWRITE = True                # False: dừng nếu tệp đích trên Drive đã tồn tại

## 2. Mount Drive và kiểm tra nguồn

In [ ]:
import os, shutil, subprocess, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Không chạy trên Colab: bỏ qua bước mount Drive.")

project_dir = Path(DRIVE_PROJECT_DIR)
source_dir = project_dir / SOURCE_NAME
if not source_dir.is_dir():
    raise FileNotFoundError(f"Không thấy thư mục {source_dir}. Kiểm tra lại DRIVE_PROJECT_DIR / shortcut trên Drive.")

archive_name = SOURCE_NAME + (".tar.gz" if COMPRESS else ".tar")
local_archive = Path(LOCAL_WORK_DIR) / archive_name
drive_archive = project_dir / archive_name

if drive_archive.exists() and not OVERWRITE:
    raise FileExistsError(f"{drive_archive} đã tồn tại; đặt OVERWRITE = True để ghi đè.")

# parser.ipynb ưu tiên data.tar.gz > data.tgz > data.tar, nên một bản nén cũ khác đuôi sẽ che bản mới.
for other in ("data.tar.gz", "data.tgz", "data.tar"):
    p = project_dir / other
    if other != archive_name and p.is_file():
        print(f"CẢNH BÁO: {p} đã tồn tại. parser.ipynb có thể dùng tệp đó thay vì {archive_name}; "
              "hãy xóa/đổi tên nếu nó đã cũ.")

print(f"Đang đếm tệp trong {source_dir} (trên Drive có thể mất vài phút)...")
started = time.perf_counter()
n_files = n_dirs = total_bytes = 0
for root, dirs, files in os.walk(source_dir):
    n_dirs += len(dirs)
    for name in files:
        n_files += 1
        try:
            total_bytes += os.lstat(os.path.join(root, name)).st_size
        except OSError:
            pass
print(f"{n_files:,} tệp, {n_dirs:,} thư mục con, {total_bytes / 2**30:.2f} GiB "
      f"(đếm trong {time.perf_counter() - started:.0f}s)")

free = shutil.disk_usage(LOCAL_WORK_DIR).free
print(f"Ổ local còn trống: {free / 2**30:.2f} GiB")
# tar cộng thêm 512 byte header mỗi mục và làm tròn khối 512 byte.
needed = total_bytes + (n_files + n_dirs) * 1024 + 2**26
if free < needed:
    raise RuntimeError(f"Ổ local không đủ chỗ: cần khoảng {needed / 2**30:.2f} GiB.")

## 3. Tạo tệp tar trên ổ local

In [ ]:
local_archive.unlink(missing_ok=True)

cmd = ["tar", "-c", "-f", str(local_archive), "-C", str(project_dir)]
if COMPRESS:
    # pigz nén song song, nhanh hơn gzip nhiều; dùng gzip nếu runtime không có pigz.
    cmd += ["--use-compress-program=pigz"] if shutil.which("pigz") else ["-z"]
for pattern in EXCLUDE:
    cmd += [f"--exclude={pattern}"]
cmd += [SOURCE_NAME]
print(" ".join(cmd))

log_path = Path(LOCAL_WORK_DIR) / (archive_name + ".log")
started = time.perf_counter()
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    last_report = started
    while proc.poll() is None:
        time.sleep(1)
        if time.perf_counter() - last_report < 15:   # in tiến độ mỗi 15s
            continue
        last_report = time.perf_counter()
        done = local_archive.stat().st_size if local_archive.exists() else 0
        pct = f" (~{min(99, 100 * done / total_bytes):.0f}%)" if total_bytes and not COMPRESS else ""
        print(f"[{last_report - started:6.0f}s] {done / 2**30:.2f} GiB{pct}")
messages = log_path.read_text(errors="replace").strip()
if messages:
    print("Thông báo của tar:\n" + messages[-3000:])
if proc.returncode != 0:
    raise RuntimeError(f"tar thất bại (exit {proc.returncode}); xem {log_path}.")

tar_seconds = time.perf_counter() - started
size = local_archive.stat().st_size
print(f"Tạo xong {local_archive}: {size / 2**30:.2f} GiB trong {tar_seconds:.0f}s")

## 4. Kiểm tra tệp tar

In [ ]:
listing = subprocess.run(["tar", "-tf", str(local_archive)], capture_output=True, text=True, check=True).stdout.splitlines()
n_entries = len(listing)
n_tar_files = sum(1 for e in listing if not e.endswith("/"))
print(f"Tar có {n_entries:,} mục ({n_tar_files:,} tệp). Nguồn: {n_files:,} tệp.")
print("Vài mục đầu:", *listing[:5], sep="\n  ")

if not all(e == SOURCE_NAME + "/" or e.startswith(SOURCE_NAME + "/") for e in listing):
    raise RuntimeError("Tar có mục nằm ngoài thư mục gốc dự kiến.")
if not EXCLUDE and n_tar_files != n_files:
    raise RuntimeError("Số tệp trong tar khác số tệp nguồn; có thể dữ liệu thay đổi trong lúc nén. Chạy lại mục 2–4.")
print("OK")

## 5. Chép lên Drive

In [ ]:
partial = drive_archive.with_name(drive_archive.name + ".partial")
partial.unlink(missing_ok=True)
print(f"Chép {local_archive} → {drive_archive} ...")
started = time.perf_counter()
shutil.copyfile(local_archive, partial)
if partial.stat().st_size != local_archive.stat().st_size:
    partial.unlink(missing_ok=True)
    raise RuntimeError("Kích thước bản chép trên Drive không khớp; thử chạy lại ô này.")
os.replace(partial, drive_archive)
print(f"Xong sau {time.perf_counter() - started:.0f}s: {drive_archive} ({drive_archive.stat().st_size / 2**30:.2f} GiB)")
print("Drive đồng bộ ngầm; đợi vài phút trước khi tắt runtime để chắc tệp đã lên hẳn Drive.")

## 6. (Tùy chọn) Dọn ổ local hoặc tải về máy

- Xóa bản local để giải phóng chỗ: chạy ô dưới.
- Tải về máy: với tệp lớn (vài GiB), tải trực tiếp từ Google Drive trên trình duyệt ổn định hơn `files.download`.

In [ ]:
# local_archive.unlink(missing_ok=True)

# from google.colab import files
# files.download(str(local_archive))